# Análisis exploratorio de datos — anuncios de vehículos

Este notebook realiza un análisis exploratorio básico del archivo `vehicles_us.csv` usando **pandas** y **Plotly**.

Objetivos:

1. Cargar y revisar la estructura del dataset.
2. Identificar valores ausentes y posibles valores atípicos.
3. Crear visualizaciones interactivas para explorar precio, kilometraje, tipo de vehículo, condición, antigüedad y fecha de publicación.


In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# En VS Code normalmente fig.show() renderiza correctamente.
# Si alguna gráfica no aparece, prueba ejecutar: pio.renderers.default = "browser"


## 1. Carga del dataset

El notebook está guardado en `notebooks/`, por eso se intenta cargar el CSV desde `../vehicles_us.csv`. También se incluye una ruta alternativa por si VS Code ejecuta el notebook desde la raíz del proyecto.


In [ ]:
csv_candidates = [
    Path("../vehicles_us.csv"),
    Path("vehicles_us.csv"),
    Path.cwd() / "../vehicles_us.csv",
    Path.cwd() / "vehicles_us.csv",
]

csv_path = next((path for path in csv_candidates if path.exists()), None)

if csv_path is None:
    raise FileNotFoundError(
        "No se encontró vehicles_us.csv. "
        "Verifica que el notebook esté en notebooks/ y que el CSV esté en la raíz del proyecto."
    )

df = pd.read_csv(csv_path)
print(f"Archivo cargado desde: {csv_path.resolve()}")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]:,}")
df.head()


## 2. Revisión inicial

Se revisan tipos de datos, estadísticas descriptivas y valores ausentes.


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


In [ ]:
missing = (
    df.isna()
    .sum()
    .reset_index(name="n_missing")
    .rename(columns={"index": "column"})
)
missing["pct_missing"] = (missing["n_missing"] / len(df) * 100).round(2)
missing.sort_values("pct_missing", ascending=False)


In [ ]:
missing_plot = missing[missing["n_missing"] > 0].sort_values("pct_missing", ascending=True)

fig = px.bar(
    missing_plot,
    x="pct_missing",
    y="column",
    orientation="h",
    text="pct_missing",
    title="Porcentaje de valores ausentes por columna",
    labels={"pct_missing": "% de valores ausentes", "column": "Columna"},
)
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_layout(yaxis_title=None)
fig.show()


## 3. Preparación ligera para EDA

No se hará una limpieza agresiva. Solo se ajustan tipos de datos y se crean variables útiles para el análisis:

- `date_posted`: fecha en formato datetime.
- `is_4wd`: se interpreta `NaN` como `0`, porque en este dataset `1` indica que el vehículo tiene tracción 4x4 y el valor faltante suele representar ausencia de 4x4.
- `vehicle_age`: antigüedad aproximada al momento de publicación.
- `month_posted`: mes de publicación.
- `brand`: primera palabra del campo `model`, usada como aproximación de marca.


In [ ]:
data = df.copy()

data["date_posted"] = pd.to_datetime(data["date_posted"], errors="coerce")
data["is_4wd"] = data["is_4wd"].fillna(0).astype(int)
data["model_year"] = data["model_year"].astype("Int64")
data["cylinders"] = data["cylinders"].astype("Int64")

data["vehicle_age"] = data["date_posted"].dt.year - data["model_year"]
data["month_posted"] = data["date_posted"].dt.to_period("M").astype(str)
data["brand"] = data["model"].str.split().str[0]

data.head()


In [ ]:
# Umbrales útiles para visualizar sin que los outliers dominen las gráficas.
price_p99 = data["price"].quantile(0.99)
odometer_p99 = data["odometer"].quantile(0.99)

print(f"Precio p99: ${price_p99:,.0f}")
print(f"Odómetro p99: {odometer_p99:,.0f} millas")


## 4. Distribución de precios

El precio suele tener sesgo a la derecha por valores muy altos. Para una visualización más útil, se limita el gráfico al percentil 99.


In [ ]:
price_data = data[data["price"] <= price_p99]

fig = px.histogram(
    price_data,
    x="price",
    nbins=70,
    title=f"Distribución de precios hasta p99 (${price_p99:,.0f})",
    labels={"price": "Precio", "count": "Número de anuncios"},
)
fig.update_layout(bargap=0.03)
fig.show()


## 5. Precio por condición del vehículo

Esta gráfica permite comparar si los vehículos en mejor condición tienden a publicarse con precios más altos.


In [ ]:
condition_order = ["salvage", "fair", "good", "excellent", "like new", "new"]

fig = px.box(
    price_data,
    x="condition",
    y="price",
    category_orders={"condition": condition_order},
    title="Distribución de precios por condición del vehículo",
    labels={"condition": "Condición", "price": "Precio"},
    points=False,
)
fig.show()


## 6. Precio y kilometraje

Se usa una muestra de datos para mantener la gráfica ágil. Se excluyen valores extremos de precio y odómetro usando el percentil 99.


In [ ]:
scatter_data = data[
    (data["price"] <= price_p99)
    & (data["odometer"] <= odometer_p99)
    & data["odometer"].notna()
].copy()

scatter_sample = scatter_data.sample(
    n=min(6_000, len(scatter_data)),
    random_state=42,
)

fig = px.scatter(
    scatter_sample,
    x="odometer",
    y="price",
    color="condition",
    hover_data=["model", "model_year", "type", "fuel", "transmission"],
    title="Relación entre kilometraje y precio",
    labels={"odometer": "Odómetro / millas", "price": "Precio", "condition": "Condición"},
)
fig.show()


## 7. Anuncios por tipo de vehículo

Esta gráfica muestra qué tipos de vehículos aparecen con mayor frecuencia en el dataset.


In [ ]:
type_counts = (
    data["type"]
    .value_counts()
    .reset_index()
)
type_counts.columns = ["type", "n_ads"]

fig = px.bar(
    type_counts,
    x="type",
    y="n_ads",
    title="Número de anuncios por tipo de vehículo",
    labels={"type": "Tipo", "n_ads": "Número de anuncios"},
    text="n_ads",
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_tickangle=-45)
fig.show()


## 8. Precio por tipo de vehículo

Se compara la distribución de precios entre tipos de vehículos, excluyendo el 1% superior de precios para mejorar la lectura.


In [ ]:
fig = px.box(
    price_data,
    x="type",
    y="price",
    title="Distribución de precios por tipo de vehículo",
    labels={"type": "Tipo", "price": "Precio"},
    points=False,
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()


## 9. Modelos más frecuentes

Se revisan los 15 modelos con más anuncios.


In [ ]:
top_models = (
    data["model"]
    .value_counts()
    .head(15)
    .reset_index()
)
top_models.columns = ["model", "n_ads"]

fig = px.bar(
    top_models.sort_values("n_ads", ascending=True),
    x="n_ads",
    y="model",
    orientation="h",
    title="Top 15 modelos con más anuncios",
    labels={"n_ads": "Número de anuncios", "model": "Modelo"},
    text="n_ads",
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title=None)
fig.show()


## 10. Antigüedad del vehículo y precio

La variable `vehicle_age` permite analizar si los autos más antiguos tienden a tener precios menores. Se filtran edades no negativas y precios hasta p99.


In [ ]:
age_price = data[
    data["vehicle_age"].notna()
    & (data["vehicle_age"] >= 0)
    & (data["vehicle_age"] <= 40)
    & (data["price"] <= price_p99)
].copy()

fig = px.scatter(
    age_price.sample(n=min(6_000, len(age_price)), random_state=42),
    x="vehicle_age",
    y="price",
    color="type",
    hover_data=["model", "model_year", "odometer", "condition"],
    title="Relación entre antigüedad del vehículo y precio",
    labels={"vehicle_age": "Antigüedad aproximada / años", "price": "Precio", "type": "Tipo"},
)
fig.show()


## 11. Anuncios por mes de publicación

Se revisa la cantidad de anuncios publicados por mes.


In [ ]:
monthly_ads = (
    data.groupby("month_posted")
    .size()
    .reset_index(name="n_ads")
    .sort_values("month_posted")
)

fig = px.line(
    monthly_ads,
    x="month_posted",
    y="n_ads",
    markers=True,
    title="Cantidad de anuncios por mes de publicación",
    labels={"month_posted": "Mes de publicación", "n_ads": "Número de anuncios"},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()


## 12. Tiempo que los anuncios permanecen publicados

Se analiza la distribución de `days_listed`, que indica cuántos días permaneció activo cada anuncio.


In [ ]:
days_p99 = data["days_listed"].quantile(0.99)
days_data = data[data["days_listed"] <= days_p99]

fig = px.histogram(
    days_data,
    x="days_listed",
    nbins=60,
    title=f"Distribución de días publicados hasta p99 ({days_p99:.0f} días)",
    labels={"days_listed": "Días publicados", "count": "Número de anuncios"},
)
fig.update_layout(bargap=0.03)
fig.show()


## 13. Comparación 4x4 vs no 4x4

Se compara el precio de vehículos con y sin tracción 4x4, usando `is_4wd` transformado a variable categórica.


In [ ]:
fourwd_data = price_data.copy()
fourwd_data["is_4wd_label"] = fourwd_data["is_4wd"].map({0: "No 4x4", 1: "4x4"})

fig = px.box(
    fourwd_data,
    x="is_4wd_label",
    y="price",
    title="Distribución de precios: vehículos 4x4 vs no 4x4",
    labels={"is_4wd_label": "Tipo de tracción", "price": "Precio"},
    points=False,
)
fig.show()


## 14. Conclusiones iniciales

Observaciones para continuar el análisis:

- El dataset contiene más de 50 mil anuncios de vehículos usados.
- Hay columnas con valores ausentes relevantes: `is_4wd`, `paint_color`, `odometer`, `cylinders` y `model_year`.
- `price` y `odometer` presentan valores extremos; conviene usar percentiles o filtros al visualizar.
- Para el desarrollo de la aplicación web, las visualizaciones más útiles probablemente serán:
  - Histograma de precios.
  - Dispersión precio vs odómetro.
  - Comparación de precio por tipo de vehículo.
  - Conteo de anuncios por tipo, modelo o condición.
